# Application on Dataset - ConceptNet Context

In [ ]:
# ==============================================================================
# ETAPA 1: INSTALAÇÃO, IMPORTS E CONFIGURAÇÃO
# ==============================================================================

In [ ]:
!pip install -q -U google-generativeai pandas scikit-learn tqdm

In [ ]:
import pandas as pd
import google.generativeai as genai
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import re
import time
import random
from google.colab import drive

In [ ]:
# --- Configuração da API Key e Montagem do Drive ---
try:
    # Substitua 'SUA_API_KEY' pela sua chave ou use o Colab Secrets
    GOOGLE_API_KEY = ''
    genai.configure(api_key=GOOGLE_API_KEY)
    print("API do Gemini configurada.")
except Exception as e:
    print(f"Chave de API não encontrada ou inválida. Erro: {e}")

In [ ]:
try:
    drive.mount('/content/drive')
except:
    print("Drive já montado ou erro ao montar.")

In [ ]:
# ==============================================================================
# ETAPA 2: CARREGAMENTO DOS DADOS (Adaptado para o dataset ABSRCT)
# ==============================================================================
PROMPTS_FILE_PATH = '//content/drive/My Drive/.../AbstRT_prompts.csv' # Usando o arquivo gerado anteriormente

try:
    df = pd.read_csv(PROMPTS_FILE_PATH)
    print(f"Arquivo '{PROMPTS_FILE_PATH}' carregado com sucesso. Total de {len(df)} linhas.")
except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{PROMPTS_FILE_PATH}' não encontrado. Certifique-se de que ele foi gerado na etapa anterior.")
    df = None

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.subject.value_counts()

In [ ]:
# ==============================================================================
# ETAPA 3: FUNÇÕES DE PROCESSAMENTO E CHAMADA À API (VERSÃO CORRIGIDA)
# ==============================================================================

def _call_gemini_api_with_retry(prompt, model):
    """
    Função auxiliar que chama a API com lógica de nova tentativa.
    O prompt já deve conter a instrução completa.
    """
    for attempt in range(2): # Tenta até 2 vezes
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if "ATTACK" in answer:
                return "ATTACK"
            elif "SUPPORT" in answer:
                return "SUPPORT"
            else:
                return "N/A" # Resposta inesperada
        except Exception as e:
            if attempt == 1: # Se for a última tentativa
                print(f"Erro final ao processar prompt: {str(e)}")
                return "ERROR"
    return "ERROR"

In [ ]:
# ==============================================================================
# BLOCO 1: DEFINIÇÃO DAS FUNÇÕES (EXPERIMENTOS 1-3)
# ==============================================================================

def get_sentiment(text, model):
    """Chama o modelo Gemini para classificar o sentimento de um texto."""
    prompt = f"Analyze the sentiment of the following text. Respond ONLY with the word POSITIVE, NEGATIVE, or NEUTRAL.\n\nText: \"{text}\"\n\nSentiment:"
    # Reutilizando a função de retry que você preferiu
    return _call_gemini_api_with_retry(prompt, model)


def process_row_basic_experiments(row, model):
    """Processa uma única linha, obtendo respostas para os três cenários iniciais."""
    # Cenário 1: Prompt Básico (sem contexto)
    prompt_without_context = (
        f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
        f"Question: The argument \"{row['argument']}\" SUPPORT or ATTACK the belief \"{row['topic']}\"?"
    )

    # Cenário 2: Prompt com Knowledge Graph
    prompt_with_kg_only = row['final_prompt'] # Já gerado no script anterior

    # Cenário 3: Prompt com KG + Sentimento
    # O sentimento já deve ter sido calculado e estar no dataframe
    arg_sentiment = row.get('argument_sentiment', 'NEUTRAL')
    topic_sentiment = row.get('topic_sentiment', 'NEUTRAL')

    sentiment_block = (
        f"Sentiment Analysis:\n"
        f"Argument has a {arg_sentiment} sentiment\n"
        f"Topic has a {topic_sentiment} sentiment"
    )

    prompt_with_sentiment = (
        f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' and sentiment analysis. "
        f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
        f"{sentiment_block}\n\n"
        f"Knowledge Graph Context: {row['knowledge_context']}\n\n"
        f"Question: The argument \"{row['argument']}\" SUPPORT or ATTACK the belief \"{row['topic']}\"?"
    )

    # Chama a API para os três cenários
    answer_without_context = _call_gemini_api_with_retry(prompt_without_context, model)
    answer_with_kg_only = _call_gemini_api_with_retry(prompt_with_kg_only, model)
    answer_with_sentiment = _call_gemini_api_with_retry(prompt_with_sentiment, model)

    # Retorna apenas as respostas, o sentimento é pré-processado
    return answer_without_context, answer_with_kg_only, answer_with_sentiment

In [ ]:
# ==============================================================================
# BLOCO 2: EXECUÇÃO DOS EXPERIMENTOS 1-3 (POR GRUPO)
# ==============================================================================

if df is not None:
    model = genai.GenerativeModel('gemini-1.5-flash')

    # Pré-calcula o sentimento para todo o dataframe de uma vez para eficiência
    print("Pré-calculando análise de sentimento para todo o dataset...")
    tqdm.pandas(desc="Analisando sentimento dos argumentos")
    df['argument_sentiment'] = df['argument'].progress_apply(lambda x: get_sentiment(x, model))
    tqdm.pandas(desc="Analisando sentimento dos tópicos")
    df['topic_sentiment'] = df['topic'].progress_apply(lambda x: get_sentiment(x, model))
    print("Análise de sentimento concluída.")

    # Pega os grupos únicos da coluna 'subject'
    subject_groups = df['subject'].unique()

    for group in subject_groups:
        print(f"\n--- Iniciando Experimentos 1-3 para o grupo: '{group}' ---")

        # Filtra o DataFrame para o grupo atual
        df_group = df[df['subject'] == group].copy()
        results = {} # Usando dicionário para manter os índices originais

        with ThreadPoolExecutor(max_workers=10) as executor:
            # Mapeia cada future ao seu índice original no DataFrame
            futures = {executor.submit(process_row_basic_experiments, row, model): index for index, row in df_group.iterrows()}

            for future in tqdm(as_completed(futures), total=len(df_group), desc=f"Avaliando '{group}' (Cenários 1-3)"):
                original_index = futures[future]
                try:
                    results[original_index] = future.result()
                except Exception as e:
                    print(f"Erro na thread para o índice {original_index}: {e}")
                    results[original_index] = ("ERROR", "ERROR", "ERROR")

        # Mapeia os resultados de volta para as colunas corretas no DataFrame principal 'df'
        # Usando .loc para garantir que os dados sejam inseridos nos índices corretos
        res_df = pd.DataFrame.from_dict(results, orient='index', columns=['gemini_without_context', 'gemini_with_context', 'gemini_with_context_sentiment'])
        df.loc[res_df.index, res_df.columns] = res_df

    print("\n--- Processamento dos Experimentos 1-3 concluído para todos os grupos. ---")
    display(df.head())
else:
    print("DataFrame 'df' não encontrado.")

In [ ]:
# ==============================================================================
# BLOCO 3: DEFINIÇÃO DA FUNÇÃO (EXPERIMENTOS 4-6)
# ==============================================================================

def process_row_advanced_experiments(row, model):
    """Processa uma única linha para os três cenários avançados."""
    # --- Prompt 4: Chain-of-Thought (CoT) ---
    cot_template = (
        "Reason in the following way:\n"
        "1. Analyze the core assertion of the argument and the topic.\n"
        "2. Does the argument provide evidence, data, or reasoning that directly confirms or contradicts the topic?\n"
        "3. Based on this relationship, conclude if the argument supports or attacks the topic.\n"
        "Conclusion:"
    )
    # Adaptação para garantir que a pergunta final não seja duplicada
    base_prompt = row['final_prompt'].split('Question:')[0]
    question = f"Question: The argument \"{row['argument']}\" SUPPORT or ATTACK the belief \"{row['topic']}\"?"
    prompt_cot = f"{base_prompt}{question}\n\n{cot_template}"

    # --- Prompt 5: Few-Shot (com 2 exemplos) ---
    example_1 = (
        "Argument: \"In the context of chronic physical illness, such as breast cancer, depression is associated with increased morbidity, longer periods of hospitalization, and greater overall disability.\"\n"
        "Belief: \"Outpatient treatment with suramin plus HC is well tolerated and provides moderate palliative benefit and delay in disease progression for patients with symptomatic HRPC.\"\n"
        "Answer: SUPPORT"
    )
    example_2 = (
        "Argument: \"Older patients, even if fit, are often considered incapable of tolerating platinum-based systemic therapy.\"\n"
        "Belief: \"Advanced age alone should not preclude appropriate NSCLC treatment.\"\n"
        "Answer: ATTACK"
    )
    few_shot_header = f"Here are some examples:\n\n[EXAMPLE 1]\n{example_1}\n\n[EXAMPLE 2]\n{example_2}\n\n---\n\nNow, complete the following task:\n"
    prompt_few_shot = (
        f"{few_shot_header}"
        f"Argument: \"{row['argument']}\"\n"
        f"Belief: \"{row['topic']}\"\n"
        f"Answer:"
    )

    # --- Prompt 6: CoT + Few-Shot ---
    prompt_cot_few_shot = f"{few_shot_header}{prompt_cot}"

    # Chama a API
    answer_cot = _call_gemini_api_with_retry(prompt_cot, model)
    answer_few_shot = _call_gemini_api_with_retry(prompt_few_shot, model)
    answer_cot_few_shot = _call_gemini_api_with_retry(prompt_cot_few_shot, model)

    return answer_cot, answer_few_shot, answer_cot_few_shot

In [ ]:
# ==============================================================================
# BLOCO 4: EXECUÇÃO DOS EXPERIMENTOS 4-6 (POR GRUPO)
# ==============================================================================

if df is not None:
    # A variável 'model' já deve existir da célula anterior

    subject_groups = df['subject'].unique()

    for group in subject_groups:
        print(f"\n--- Iniciando Experimentos 4-6 para o grupo: '{group}' ---")
        df_group = df[df['subject'] == group].copy()
        results_advanced = {}

        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = {executor.submit(process_row_advanced_experiments, row, model): index for index, row in df_group.iterrows()}

            for future in tqdm(as_completed(futures), total=len(df_group), desc=f"Avaliando '{group}' (Cenários 4-6)"):
                original_index = futures[future]
                try:
                    results_advanced[original_index] = future.result()
                except Exception as e:
                    print(f"Erro na thread para o índice {original_index}: {e}")
                    results_advanced[original_index] = ("ERROR", "ERROR", "ERROR")

        # Mapeia os resultados de volta para o DataFrame principal 'df'
        res_adv_df = pd.DataFrame.from_dict(results_advanced, orient='index', columns=['gemini_cot', 'gemini_few_shot', 'gemini_cot_few_shot'])
        df.loc[res_adv_df.index, res_adv_df.columns] = res_adv_df

    # Salva o arquivo final com todos os resultados de todos os grupos
    output_filename_final = '/content/drive/MyDrive/ABSRCT_evaluated_all_ways.csv'
    df.to_csv(output_filename_final, index=False, encoding='utf-8')
    print(f"\n--- Processamento final concluído. Resultados salvos em '{output_filename_final}' ---")
    display(df.head())
else:
    print("DataFrame 'df' não encontrado.")

In [ ]:
# ==============================================================================
# BLOCO 5: ANÁLISE FINAL E MÉTRICAS (POR GRUPO)
# ==============================================================================

def analyze_results_by_group(results_df):
    """Calcula e exibe métricas para todos os 6 cenários, separados por grupo."""

    subject_groups = results_df['subject'].unique()

    for group in subject_groups:
        print("\n" + "="*80)
        print(f" ANÁLISE DE DESEMPENHO PARA O GRUPO: '{group}'")
        print("="*80)

        group_df = results_df[results_df['subject'] == group]
        y_true = group_df['ground_truth_stance']

        scenarios = {
            "1. Sem Contexto": "gemini_without_context",
            "2. Com Contexto KG": "gemini_with_context",
            "3. Com KG + Sentimento": "gemini_with_context_sentiment",
            "4. Chain-of-Thought (CoT)": "gemini_cot",
            "5. Few-Shot": "gemini_few_shot",
            "6. CoT + Few-Shot": "gemini_cot_few_shot"
        }

        labels = sorted(y_true.dropna().unique())

        for name, col in scenarios.items():
            if col in group_df.columns:
                y_pred = group_df[col]
                valid_mask = group_df[col].isin(labels)

                if valid_mask.sum() > 0:
                    acc = accuracy_score(y_true, y_pred)
                    f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average='weighted', labels=labels, zero_division=0)

                    print(f"\n[Resultados para: {name}]")
                    print(f"  Acurácia Geral: {acc:.4f}")
                    print(f"  F1-Score (Weighted): {f1:.4f}")

                    cm = confusion_matrix(y_true[valid_mask], y_pred[valid_mask], labels=labels)
                    plt.figure(figsize=(5, 3))
                    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
                    plt.title(f'Matriz de Confusão - {name} ({group})')
                    plt.ylabel('Verdadeiro')
                    plt.xlabel('Predito')
                    plt.show()
                else:
                    print(f"\n[Resultados para: {name}] - Nenhuma resposta válida encontrada.")
            else:
                print(f"\nColuna '{col}' não encontrada para o cenário '{name}'.")

        # --- ADIÇÃO: Imprime um exemplo de prompt avançado para o grupo ---
        print("\n--- Exemplo de Prompt (Few-Shot) para este grupo ---")
        if not group_df.empty:
            example_row = group_df.iloc[0] # Pega a primeira linha do grupo
            example_1 = (
                "Argument: \"In the context of chronic physical illness, such as breast cancer, depression is associated with increased morbidity, longer periods of hospitalization, and greater overall disability.\"\n"
                "Belief: \"Outpatient treatment with suramin plus HC is well tolerated and provides moderate palliative benefit and delay in disease progression for patients with symptomatic HRPC.\"\n"
                "Answer: SUPPORT"
            )
            example_2 = (
                "Argument: \"Older patients, even if fit, are often considered incapable of tolerating platinum-based systemic therapy.\"\n"
                "Belief: \"Advanced age alone should not preclude appropriate NSCLC treatment.\"\n"
                "Answer: ATTACK"
            )
            few_shot_header = f"Here are some examples:\n\n[EXAMPLE 1]\n{example_1}\n\n[EXAMPLE 2]\n{example_2}\n\n---\n\nNow, complete the following task:\n"
            prompt_example = (
                f"{few_shot_header}"
                f"Argument: \"{example_row['argument']}\"\n"
                f"Belief: \"{example_row['topic']}\"\n"
                f"Answer:"
            )
            print(prompt_example)
        else:
            print("Não há dados para gerar um exemplo.")


# --- Execução da Análise Final ---
if 'gemini_cot_few_shot' in df.columns:
    analyze_results_by_group(df)
else:
    print("\nAnálise final não pode ser executada. Execute as etapas anteriores.")